# REPA × CIFAR-10 × MedDINOv3

Trains **SiT-S/8** on CIFAR-10 using **MedDINOv3 ViT-Base** as the representation encoder.

**Requirements (Kaggle datasets attached to this notebook):**
- `nikiboura/meddinov3` — contains MedDINOv3 code + checkpoint

All code runs from the cloned `nikiboura/REPA` fork. No local file edits.

## 1. Install dependencies

In [ ]:
%%bash
pip install -q accelerate diffusers timm einops

## 2. Clone REPA fork

In [ ]:
%%bash
cd /kaggle/working
if [ ! -d "REPA" ]; then
    git clone https://github.com/nikiboura/REPA.git REPA
fi
echo "REPA ready. Commit: $(git -C REPA rev-parse --short HEAD)"

## 3. Set paths

In [ ]:
import os

REPA_DIR   = '/kaggle/working/REPA'
DATA_DIR   = '/kaggle/working/data/cifar10_256'
MEDDINOV3  = '/kaggle/input/datasets/nikiboura/meddinov3-minimal/meddinov3_minimal'
CKPT_PATH  = '/kaggle/input/datasets/nikiboura/meddinov3-minimal/meddinov3_minimal/checkpoints/model.pth'

os.environ['MEDDINOV3_PATH'] = MEDDINOV3
os.environ['MEDDINOV3_CKPT'] = CKPT_PATH

os.makedirs(DATA_DIR, exist_ok=True)

print(f'MedDINOv3 exists : {os.path.isdir(MEDDINOV3)}')
print(f'Checkpoint exists: {os.path.isfile(CKPT_PATH)}')

## 4. Prepare CIFAR-10
Downloads CIFAR-10, resizes to 256×256, VAE-encodes each image, saves latents as `.npy`.

In [ ]:
import subprocess, sys

cmd = [
    sys.executable, '/kaggle/working/REPA/prepare_cifar10.py',
    '--out-dir', '/kaggle/working/data/cifar10_256',
    '--resolution', '256',
    '--max-samples', '5000',
]
process = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in process.stdout:
    print(line, end='', flush=True)
process.wait()
print('Exit code:', process.returncode)


## 5. Train
Runs 1000 steps as a quick correctness check. Checkpoints saved to `/kaggle/working/results/`.

In [ ]:
import subprocess, os, re
from tqdm.notebook import tqdm

env = os.environ.copy()
env['MEDDINOV3_PATH'] = '/kaggle/input/datasets/nikiboura/meddinov3-minimal/meddinov3_minimal'
env['MEDDINOV3_CKPT'] = '/kaggle/input/datasets/nikiboura/meddinov3-minimal/meddinov3_minimal/checkpoints/model.pth'

cmd = [
    'accelerate', 'launch',
    '--mixed_precision', 'fp16',
    '--num_processes', '1',
    '/kaggle/working/REPA/train.py',
    '--exp-name',            'repa-sits8-meddinov3-cifar10',
    '--output-dir',          '/kaggle/working/results',
    '--report-to',           'none',
    '--model',               'SiT-S/8',
    '--num-classes',         '10',
    '--encoder-depth',       '8',
    '--enc-type',            'meddinov3-vit-b',
    '--data-dir',            '/kaggle/working/data/cifar10_256',
    '--resolution',          '256',
    '--batch-size',          '32',
    '--num-workers',         '2',
    '--epochs',              '200',
    '--max-train-steps',     '5000',
    '--learning-rate',       '1e-4',
    '--mixed-precision',     'fp16',
    '--checkpointing-steps', '5000',
    '--sampling-steps',      '5000',
    '--proj-coeff',          '0.5',
    '--cfg-prob',            '0.1',
    '--path-type',           'linear',
]

process = subprocess.Popen(cmd, cwd='/kaggle/working/REPA', env=env,
                           stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)

pbar = tqdm(total=5000, desc='Training')
last_step = 0
for line in process.stdout:
    m = re.search(r'Steps:\s*(\d+)/\d+.*?loss=([\d.]+).*?proj_loss=(-?[\d.]+)', line)
    if m:
        step = int(m.group(1))
        if step > last_step:
            pbar.update(step - last_step)
            pbar.set_postfix({'loss': m.group(2), 'proj': m.group(3)})
            last_step = step
    else:
        stripped = line.strip()
        if stripped:
            print(stripped, flush=True)
pbar.close()
process.wait()
print('Exit code:', process.returncode)


## 6. Generate samples

In [ ]:
import subprocess, os, glob

env = os.environ.copy()
env['MEDDINOV3_PATH'] = '/kaggle/input/datasets/nikiboura/meddinov3-minimal/meddinov3_minimal'
env['MEDDINOV3_CKPT'] = '/kaggle/input/datasets/nikiboura/meddinov3-minimal/meddinov3_minimal/checkpoints/model.pth'

ckpts = sorted(glob.glob('/kaggle/working/results/repa-sits8-meddinov3-cifar10/checkpoints/*.pt'))
print('Using checkpoint:', ckpts[-1])

cmd = [
    'torchrun', '--nproc_per_node=1',
    '/kaggle/working/REPA/generate.py',
    '--model',                'SiT-S/8',
    '--ckpt',                 ckpts[-1],
    '--encoder-depth',        '8',
    '--num-classes',          '10',
    '--projector-embed-dims', '768',
    '--per-proc-batch-size',  '16',
    '--num-fid-samples',      '64',
    '--path-type',            'linear',
    '--mode',                 'ode',
    '--num-steps',            '50',
    '--cfg-scale',            '1.0',
    '--resolution',           '256',
    '--vae',                  'mse',
]

result = subprocess.run(cmd, cwd='/kaggle/working/REPA', env=env, text=True)
print('Exit code:', result.returncode)


## 7. Show generated images

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import glob

npz_files = sorted(glob.glob('/kaggle/working/REPA/samples/**/*.npz', recursive=True))
if not npz_files:
    print('No samples found.')
else:
    data = np.load(npz_files[-1])
    imgs = data[data.files[0]]

    fig, axes = plt.subplots(4, 4, figsize=(8, 8))
    for i, ax in enumerate(axes.flat):
        ax.imshow(imgs[i].astype('uint8'))
        ax.axis('off')
    plt.tight_layout()
    plt.show()
    print(f'From: {npz_files[-1]}')
